# Name: Lin Qizhou
# Key insights / takeaways (to be finalized after audit):
# 1) This audit uses true calendar day counts (including leap day) rather than “30-day months”.
# 2) Interest is computed daily: Interest = Balance * (0.12/365) * Days_Elapsed between payment dates.
# 3) We will validate the lender’s interest by recomputing interest over irregular payment intervals and producing an event-by-event ledger.


In [1]:
from datetime import date
import pandas as pd

# Loan parameters
principal = 100_000.00
annual_rate = 0.12
daily_rate = annual_rate / 365

start_date = date(2024, 1, 1)  # Leap year

# Payment log (irregular dates)
payments = [
    {"date": date(2024, 1, 31), "payment": 5_000.00},
    {"date": date(2024, 2, 29), "payment": 5_000.00},  # Leap Day
    {"date": date(2024, 3, 31), "payment": 5_000.00},
]

df = pd.DataFrame(payments).sort_values("date").reset_index(drop=True)
df


,date,payment
0,2024-01-31,5000.0
1,2024-02-29,5000.0
2,2024-03-31,5000.0


In [2]:
# Compute days elapsed using Python date difference: (end - start).days
# Convention here: days between dates, excluding the start date (standard timedelta days).
prev_dates = [start_date] + df["date"].tolist()[:-1]
df["prev_date"] = prev_dates
df["days_elapsed"] = [(d - p).days for d, p in zip(df["date"], df["prev_date"])]

df[["prev_date", "date", "days_elapsed", "payment"]]


,prev_date,date,days_elapsed,payment
0,2024-01-01,2024-01-31,30,5000.0
1,2024-01-31,2024-02-29,29,5000.0
2,2024-02-29,2024-03-31,31,5000.0


In [3]:
# Step 3 — compute accrued interest for each interval, BEFORE applying the payment

balance = principal
interests = []

for i, row in df.iterrows():
    days = int(row["days_elapsed"])
    interest = balance * daily_rate * days
    interests.append(interest)

    # Note: payment allocation (interest vs principal) will be done in the next step.
    # Here we only record the interest accrued on the opening balance of the interval.

df["opening_balance"] = [principal] + [None]*(len(df)-1)
# Fill opening_balance iteratively for display clarity
opening_balances = []
balance = principal
for i, row in df.iterrows():
    opening_balances.append(balance)
    # do NOT update balance here (we haven’t allocated payment yet)
df["opening_balance"] = opening_balances

df["accrued_interest"] = interests

df[["prev_date", "date", "days_elapsed", "opening_balance", "accrued_interest", "payment"]]


,prev_date,date,days_elapsed,opening_balance,accrued_interest,payment
0,2024-01-01,2024-01-31,30,100000.0,986.301370,5000.0
1,2024-01-31,2024-02-29,29,100000.0,953.424658,5000.0
2,2024-02-29,2024-03-31,31,100000.0,1019.178082,5000.0


In [4]:
# Step 4 — Apply each payment: interest first, then principal reduction

ledger_rows = []

balance = principal

for i, row in df.iterrows():
    prev_date = row["prev_date"]
    pay_date  = row["date"]
    days      = int(row["days_elapsed"])
    payment   = float(row["payment"])

    interest = balance * daily_rate * days

    interest_paid = min(payment, interest)
    principal_paid = max(0.0, payment - interest_paid)

    ending_balance = balance - principal_paid

    ledger_rows.append({
        "prev_date": prev_date,
        "payment_date": pay_date,
        "days_elapsed": days,
        "opening_balance": balance,
        "accrued_interest": interest,
        "payment": payment,
        "interest_paid": interest_paid,
        "principal_paid": principal_paid,
        "ending_balance": ending_balance,
    })

    # Update balance for next interval
    balance = ending_balance

ledger = pd.DataFrame(ledger_rows)

# Add running totals (useful for audit)
ledger["cum_interest"] = ledger["accrued_interest"].cumsum()
ledger["cum_principal_paid"] = ledger["principal_paid"].cumsum()
ledger["cum_payments"] = ledger["payment"].cumsum()

ledger


,prev_date,payment_date,days_elapsed,opening_balance,accrued_interest,payment,interest_paid,principal_paid,ending_balance,cum_interest,cum_principal_paid,cum_payments
0,2024-01-01,2024-01-31,30,100000.000000,986.301370,5000.0,986.301370,4013.698630,95986.301370,986.301370,4013.698630,5000.0
1,2024-01-31,2024-02-29,29,95986.301370,915.157065,5000.0,915.157065,4084.842935,91901.458435,1901.458435,8098.541565,10000.0
2,2024-02-29,2024-03-31,31,91901.458435,936.639522,5000.0,936.639522,4063.360478,87838.097957,2838.097957,12161.902043,15000.0


In [5]:
# Step 5 — Audit summary (correct calendar math) + compare to a common "30-day" mistake

def build_ledger_with_custom_days(custom_days_list):
    """
    Recompute ledger using a provided list of days_elapsed (same length as df).
    Interest = opening_balance * (annual_rate/365) * days
    Payment allocates to interest first, then principal.
    """
    rows = []
    bal = principal

    for i, row in df.iterrows():
        days = int(custom_days_list[i])
        payment = float(row["payment"])

        interest = bal * daily_rate * days
        interest_paid = min(payment, interest)
        principal_paid = max(0.0, payment - interest_paid)
        end_bal = bal - principal_paid

        rows.append({
            "payment_date": row["date"],
            "days_elapsed": days,
            "opening_balance": bal,
            "accrued_interest": interest,
            "payment": payment,
            "interest_paid": interest_paid,
            "principal_paid": principal_paid,
            "ending_balance": end_bal,
        })

        bal = end_bal

    out = pd.DataFrame(rows)
    out["total_interest"] = out["accrued_interest"].cumsum()
    out["total_principal_paid"] = out["principal_paid"].cumsum()
    out["total_payments"] = out["payment"].cumsum()
    return out

# 1) Correct ledger summary (from Step 4)
correct_total_interest = float(ledger["accrued_interest"].sum())
correct_end_balance = float(ledger["ending_balance"].iloc[-1])
correct_total_principal = float(ledger["principal_paid"].sum())
correct_total_payments = float(ledger["payment"].sum())

# 2) Wrong assumption scenario: "30 days per period" (common trap)
wrong_days = [30, 30, 30]
ledger_30 = build_ledger_with_custom_days(wrong_days)

wrong_total_interest = float(ledger_30["accrued_interest"].sum())
wrong_end_balance = float(ledger_30["ending_balance"].iloc[-1])
wrong_total_principal = float(ledger_30["principal_paid"].sum())

# 3) Differences (positive means the "30-day mistake" overcharges interest)
interest_diff = wrong_total_interest - correct_total_interest
end_balance_diff = wrong_end_balance - correct_end_balance

summary = pd.DataFrame([
    {
        "Method": "Correct (calendar days)",
        "Total Payments": correct_total_payments,
        "Total Interest": correct_total_interest,
        "Total Principal Paid": correct_total_principal,
        "Ending Balance": correct_end_balance,
    },
    {
        "Method": "Wrong example (30 days each period)",
        "Total Payments": correct_total_payments,
        "Total Interest": wrong_total_interest,
        "Total Principal Paid": wrong_total_principal,
        "Ending Balance": wrong_end_balance,
    },
    {
        "Method": "Difference (Wrong - Correct)",
        "Total Payments": 0.0,
        "Total Interest": interest_diff,
        "Total Principal Paid": wrong_total_principal - correct_total_principal,
        "Ending Balance": end_balance_diff,
    }
])

# Pretty print
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("=== Audit Summary ===")
display(summary)

print("\n=== Wrong-method ledger (30-day periods) ===")
display(ledger_30[["payment_date","days_elapsed","opening_balance","accrued_interest","payment","principal_paid","ending_balance"]])

print("\nInterpretation guide:")
print("- If Difference Total Interest is positive, the wrong method overcharges interest vs correct calendar math.")
print("- Ending Balance difference shows how the mistake propagates into a wrong remaining balance.")


=== Audit Summary ===


,Method,Total Payments,Total Interest,Total Principal Paid,Ending Balance
0,Correct (calendar days),"15,000.00","2,838.10","12,161.90","87,838.10"
1,Wrong example (30 days each period),"15,000.00","2,839.75","12,160.25","87,839.75"
2,Difference (Wrong - Correct),0.00,1.65,-1.65,1.65



=== Wrong-method ledger (30-day periods) ===


,payment_date,days_elapsed,opening_balance,accrued_interest,payment,principal_paid,ending_balance
0,2024-01-31,30,"100,000.00",986.30,"5,000.00","4,013.70","95,986.30"
1,2024-02-29,30,"95,986.30",946.71,"5,000.00","4,053.29","91,933.02"
2,2024-03-31,30,"91,933.02",906.74,"5,000.00","4,093.26","87,839.75"



Interpretation guide:
- If Difference Total Interest is positive, the wrong method overcharges interest vs correct calendar math.
- Ending Balance difference shows how the mistake propagates into a wrong remaining balance.


In [6]:
# Step 6A — Generate an English audit conclusion from computed results

correct = summary.loc[summary["Method"] == "Correct (calendar days)"].iloc[0]
diffrow = summary.loc[summary["Method"] == "Difference (Wrong - Correct)"].iloc[0]

conclusion = f"""
Audit Conclusion (Task 9)

Using true calendar day counts and the stated daily interest formula (balance × 0.12/365 × days elapsed),
the total interest accrued across the three irregular intervals is {correct['Total Interest']:.2f},
with total payments of {correct['Total Payments']:.2f}, resulting in total principal reduction of
{correct['Total Principal Paid']:.2f} and an ending balance of {correct['Ending Balance']:.2f}.

A common “date drift” mistake is to assume uniform 30-day periods. Under that wrong assumption,
the computed interest differs from the correct calendar method by {diffrow['Total Interest']:.2f}
(Wrong − Correct). A positive value indicates overcharging interest; a negative value indicates undercharging.

Therefore, any lender statement that uses fixed 30-day periods instead of true day counts would produce
a measurably different interest total and ending balance, which is exactly the kind of discrepancy this
amortization audit is designed to detect.
""".strip()

print(conclusion)


Audit Conclusion (Task 9)

Using true calendar day counts and the stated daily interest formula (balance × 0.12/365 × days elapsed),
the total interest accrued across the three irregular intervals is 2838.10,
with total payments of 15000.00, resulting in total principal reduction of
12161.90 and an ending balance of 87838.10.

A common “date drift” mistake is to assume uniform 30-day periods. Under that wrong assumption,
the computed interest differs from the correct calendar method by 1.65
(Wrong − Correct). A positive value indicates overcharging interest; a negative value indicates undercharging.

Therefore, any lender statement that uses fixed 30-day periods instead of true day counts would produce
a measurably different interest total and ending balance, which is exactly the kind of discrepancy this
amortization audit is designed to detect.


In [7]:
# Step 6B — Alternative convention: include the payment date in day count (+1 day each interval)

# current convention: days = (pay_date - prev_date).days  -> already in df["days_elapsed"]
days_exclusive = df["days_elapsed"].tolist()

# alternative convention: include payment date -> +1 each interval
days_inclusive = [d + 1 for d in days_exclusive]

ledger_inclusive = build_ledger_with_custom_days(days_inclusive)

exclusive_total_interest = float(ledger["accrued_interest"].sum())
inclusive_total_interest = float(ledger_inclusive["accrued_interest"].sum())
interest_delta = inclusive_total_interest - exclusive_total_interest

compare = pd.DataFrame([
    {"Convention": "Exclusive (timedelta days)", "Total Interest": exclusive_total_interest,
     "Ending Balance": float(ledger["ending_balance"].iloc[-1])},
    {"Convention": "Inclusive (+1 day each payment date)", "Total Interest": inclusive_total_interest,
     "Ending Balance": float(ledger_inclusive["ending_balance"].iloc[-1])},
    {"Convention": "Difference (Inclusive - Exclusive)", "Total Interest": interest_delta,
     "Ending Balance": float(ledger_inclusive["ending_balance"].iloc[-1]) - float(ledger["ending_balance"].iloc[-1])},
])

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
display(compare)


,Convention,Total Interest,Ending Balance
0,Exclusive (timedelta days),"2,838.10","87,838.10"
1,Inclusive (+1 day each payment date),"2,933.75","87,933.75"
2,Difference (Inclusive - Exclusive),95.65,95.65
